# 🕵️ Dia 40 — Pivotando infraestrutura de ameaça em um grafo

**Investigador Cyber 90 Dias** · Fase 3 · Rank Jutsu `B`

> Baseado no *Day 40* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-3-cyber-threat-intelligence/day-40.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Sua primeira versão do grafo tinha 47 nós, incluindo `padaria-do-bairro.example`, que só dividia o IP com o kit. Rafa apagou 38 com três cliques: *"Hospedagem compartilhada é vizinhança, não parentesco. Cada aresta diz qual pivô a criou e quantos resultados ele devolveu."*

Neste notebook você vai:

1. **Consultar o crt.sh em JSON via `urllib`**, com plano B offline embutido.
2. **Calcular o hash de favicon do Shodan sem `mmh3`** (MurmurHash3 em Python puro).
3. **Aplicar "contagem antes de busca"** a todos os pivôs do caso e gerar o registro de pivôs.
4. **Montar `nodes.csv` e `edges.csv`** e classificar membros × não confirmados.
5. **Desenhar o grafo em Mermaid** e gerar a camada de *Resource Development*.

🐍 Só biblioteca padrão do Python. Nada para instalar. Funciona offline: toda célula que tenta a rede tem plano B embutido. Rode as células de cima para baixo (Shift+Enter).

## 1. crt.sh em JSON (passo 1 do original)

A célula consulta o crt.sh para um **domínio público e legítimo** à sua escolha (padrão: `example.com`, que existe de verdade e é reservado pela IANA para exemplos). Ela nunca é apontada para os domínios do caso: eles são `.example`, fictícios, e no trabalho real você consultaria o crt.sh, não o domínio.

O crt.sh vive sobrecarregado e devolve página HTML de erro em vez de JSON. O original manda **esperar e tentar de novo**, sem martelar. A célula tenta no máximo duas vezes, com espera, e depois cai no **plano B**: o crt.sh do caso, embutido.

In [ ]:
#@title crt.sh { display-mode: "form" }
DOMINIO_CRT = "example.com"  #@param {type:"string"}
TENTAR_REDE = True  #@param {type:"boolean"}

import json
import time
import urllib.parse
import urllib.request

RESP = json.loads('''{
 "_aviso": "Respostas SIMULADAS e FICTÍCIAS de serviços de pivô (crt.sh, Shodan, urlscan.io) da Operação Isca (Investigador Cyber 90 Dias, Dia 40). Formatos inspirados nos reais; nenhum valor veio desses serviços. Domínios .example (RFC 2606), IPs de documentação (RFC 5737). Coerente com caso-continuo/dados/dia-30 e dia-43.",
 "consultado_em": "2026-10-13T15:00:00Z",
 "crtsh": [
  {
   "id": 90000000001,
   "issuer_name": "C=XX, O=CA Gratuita Exemplo, CN=E1",
   "common_name": "serraazul-pix.example",
   "name_value": "serraazul-pix.example\\nwww.serraazul-pix.example\\nserraazul-seguranca.example\\npainel.serraazul-pix.example",
   "entry_timestamp": "2026-10-04T19:12:41.120",
   "not_before": "2026-10-04T18:12:41",
   "not_after": "2027-01-02T18:12:40",
   "serial_number": "0aa1fict0001"
  },
  {
   "id": 90000000002,
   "issuer_name": "C=XX, O=CA Gratuita Exemplo, CN=E1",
   "common_name": "mail.serraazul-seguranca.example",
   "name_value": "mail.serraazul-seguranca.example",
   "entry_timestamp": "2026-10-08T23:58:03.881",
   "not_before": "2026-10-08T22:58:03",
   "not_after": "2027-01-06T22:58:02",
   "serial_number": "0aa1fict0002"
  },
  {
   "id": 90000000003,
   "issuer_name": "C=XX, O=CA Gratuita Exemplo, CN=E1",
   "common_name": "coopvale-pix.example",
   "name_value": "coopvale-pix.example\\nwww.coopvale-pix.example",
   "entry_timestamp": "2026-10-07T22:40:10.004",
   "not_before": "2026-10-07T21:40:10",
   "not_after": "2027-01-05T21:40:09",
   "serial_number": "0aa1fict0003"
  },
  {
   "id": 90000000004,
   "issuer_name": "C=XX, O=CA Gratuita Exemplo, CN=E1",
   "common_name": "credminas-seguranca.example",
   "name_value": "credminas-seguranca.example\\nunicoop-sul-pix.example\\nwww.credminas-seguranca.example",
   "entry_timestamp": "2026-10-08T20:05:51.310",
   "not_before": "2026-10-08T19:05:51",
   "not_after": "2027-01-06T19:05:50",
   "serial_number": "0aa1fict0004"
  },
  {
   "id": 90000000005,
   "issuer_name": "C=XX, O=CA Gratuita Exemplo, CN=E1",
   "common_name": "parking-lote-17.registrar-parking.example",
   "name_value": "parking-lote-17.registrar-parking.example\\ncoopserra-azul.example\\ndominio-estacionado-001.example\\ndominio-estacionado-002.example\\ndominio-estacionado-003.example\\ndominio-estacionado-004.example\\ndominio-estacionado-005.example\\ndominio-estacionado-006.example\\ndominio-estacionado-007.example\\ndominio-estacionado-008.example\\ndominio-estacionado-009.example\\ndominio-estacionado-010.example\\ndominio-estacionado-011.example\\ndominio-estacionado-012.example\\ndominio-estacionado-013.example\\ndominio-estacionado-014.example\\ndominio-estacionado-015.example\\ndominio-estacionado-016.example\\ndominio-estacionado-017.example\\ndominio-estacionado-018.example\\ndominio-estacionado-019.example\\ndominio-estacionado-020.example\\ndominio-estacionado-021.example\\ndominio-estacionado-022.example\\ndominio-estacionado-023.example\\ndominio-estacionado-024.example\\ndominio-estacionado-025.example\\ndominio-estacionado-026.example\\ndominio-estacionado-027.example\\ndominio-estacionado-028.example\\ndominio-estacionado-029.example\\ndominio-estacionado-030.example\\ndominio-estacionado-031.example\\ndominio-estacionado-032.example\\ndominio-estacionado-033.example\\ndominio-estacionado-034.example\\ndominio-estacionado-035.example\\ndominio-estacionado-036.example\\ndominio-estacionado-037.example\\ndominio-estacionado-038.example\\ndominio-estacionado-039.example\\ndominio-estacionado-040.example\\ndominio-estacionado-041.example\\ndominio-estacionado-042.example\\ndominio-estacionado-043.example\\ndominio-estacionado-044.example\\ndominio-estacionado-045.example\\ndominio-estacionado-046.example\\ndominio-estacionado-047.example\\ndominio-estacionado-048.example\\ndominio-estacionado-049.example\\ndominio-estacionado-050.example\\ndominio-estacionado-051.example\\ndominio-estacionado-052.example\\ndominio-estacionado-053.example\\ndominio-estacionado-054.example\\ndominio-estacionado-055.example\\ndominio-estacionado-056.example\\ndominio-estacionado-057.example\\ndominio-estacionado-058.example\\ndominio-estacionado-059.example\\ndominio-estacionado-060.example\\ndominio-estacionado-061.example\\ndominio-estacionado-062.example\\ndominio-estacionado-063.example\\ndominio-estacionado-064.example\\ndominio-estacionado-065.example\\ndominio-estacionado-066.example\\ndominio-estacionado-067.example\\ndominio-estacionado-068.example\\ndominio-estacionado-069.example\\ndominio-estacionado-070.example\\ndominio-estacionado-071.example\\ndominio-estacionado-072.example\\ndominio-estacionado-073.example\\ndominio-estacionado-074.example\\ndominio-estacionado-075.example\\ndominio-estacionado-076.example\\ndominio-estacionado-077.example\\ndominio-estacionado-078.example\\ndominio-estacionado-079.example\\ndominio-estacionado-080.example\\ndominio-estacionado-081.example\\ndominio-estacionado-082.example\\ndominio-estacionado-083.example\\ndominio-estacionado-084.example\\ndominio-estacionado-085.example\\ndominio-estacionado-086.example\\ndominio-estacionado-087.example\\ndominio-estacionado-088.example\\ndominio-estacionado-089.example\\ndominio-estacionado-090.example\\ndominio-estacionado-091.example\\ndominio-estacionado-092.example\\ndominio-estacionado-093.example\\ndominio-estacionado-094.example\\ndominio-estacionado-095.example\\ndominio-estacionado-096.example",
   "entry_timestamp": "2026-10-03T03:00:00.000",
   "not_before": "2026-10-03T02:00:00",
   "not_after": "2027-01-01T01:59:59",
   "serial_number": "0aa1fict0005"
  }
 ],
 "crtsh_padrao_nome": {
  "consulta": "%pix%.example",
  "resultados": 1287,
  "nota": "padrão de nome: pista fraca, não conta para convergência"
 },
 "shodan_host": {
  "203.0.113.77": [
   {
    "port": 443,
    "http.favicon.hash": -1029384756,
    "http.title": "Serra Azul - Regularização Pix",
    "http.html_hash": 1474329905,
    "ssl.cert.fingerprint.sha256": "9f2cfict00000000000000000000000000000000000000000000000000e41a",
    "ssl.jarm": "2ad2ad16d2ad2ad22c42d42d00000069fict"
   }
  ],
  "198.51.100.40": [
   {
    "port": 443,
    "http.favicon.hash": -1029384756,
    "http.title": "CredMinas - Regularização Pix",
    "http.html_hash": -88231407,
    "ssl.cert.fingerprint.sha256": "4b7dfict00000000000000000000000000000000000000000000000000c09e",
    "ssl.jarm": "2ad2ad16d2ad2ad22c42d42d00000069fict"
   }
  ]
 },
 "shodan_count": {
  "http.favicon.hash:-1029384756": 9,
  "http.html_hash:1474329905": 3,
  "ssl.cert.fingerprint:9f2cfict00000000000000000000000000000000000000000000000000e41a": 1,
  "ssl.jarm:2ad2ad16d2ad2ad22c42d42d00000069fict": 18544,
  "http.title:\\"Regularização de Chave Pix\\"": 214,
  "http.favicon.hash:116323821": 412907
 },
 "shodan_search": {
  "http.favicon.hash:-1029384756": [
   [
    "203.0.113.77",
    443,
    [
     "serraazul-pix.example",
     "coopvale-pix.example"
    ],
    "Serra Azul - Regularização Pix"
   ],
   [
    "198.51.100.40",
    443,
    [
     "credminas-seguranca.example",
     "unicoop-sul-pix.example"
    ],
    "CredMinas - Regularização Pix"
   ],
   [
    "192.0.2.150",
    443,
    [],
    "Painel de teste"
   ]
  ],
  "http.html_hash:1474329905": [
   [
    "203.0.113.77",
    443,
    [
     "serraazul-pix.example",
     "coopvale-pix.example"
    ],
    "Serra Azul - Regularização Pix"
   ]
  ]
 },
 "urlscan": {
  "page.ip:\\"203.0.113.77\\"": {
   "total": 41,
   "amostra": [
    [
     "2026-10-05T10:42:18Z",
     "serraazul-pix.example",
     "Serra Azul - Regularização Pix"
    ],
    [
     "2026-10-08T09:03:51Z",
     "coopvale-pix.example",
     "CoopVale - Regularização Pix"
    ],
    [
     "2026-09-12T08:00:00Z",
     "padaria-do-bairro.example",
     "Padaria do Bairro"
    ],
    [
     "2026-08-30T14:00:00Z",
     "escola-de-musica.example",
     "Escola de Música"
    ]
   ]
  },
  "hash:c6a500d420b2dd0197d73e8fa5ebe43f81ebfa3e86e56b91a9e909a4beae08b6": {
   "total": 2,
   "amostra": [
    [
     "2026-10-05T10:42:18Z",
     "serraazul-pix.example",
     "203.0.113.77"
    ],
    [
     "2026-10-08T09:03:51Z",
     "coopvale-pix.example",
     "203.0.113.77"
    ]
   ]
  },
  "hash:e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855": {
   "total": 10000,
   "amostra": [],
   "nota": "SHA-256 do arquivo vazio"
  },
  "page.domain:credminas-seguranca.example": {
   "total": 1,
   "amostra": [
    [
     "2026-10-09T15:20:00Z",
     "credminas-seguranca.example",
     "198.51.100.40"
    ]
   ],
   "nota": "a varredura pública guardou um isca.js com novo SHA-256 e o comentário '// pescador'"
  },
  "page.domain:cooperouro-regulariza.example": {
   "total": 1,
   "amostra": [
    [
     "2026-10-11T18:02:00Z",
     "cooperouro-regulariza.example",
     "192.0.2.77"
    ]
   ],
   "nota": "título igual, nenhum recurso em comum com o kit"
  }
 },
 "vt_resolutions": {
  "serraazul-pix.example": [
   [
    "2026-10-04T19:30:11Z",
    "203.0.113.77"
   ]
  ],
  "serraazul-seguranca.example": [
   [
    "2026-10-05T08:01:44Z",
    "203.0.113.78"
   ]
  ],
  "coopvale-pix.example": [
   [
    "2026-10-07T22:15:00Z",
    "203.0.113.77"
   ]
  ],
  "credminas-seguranca.example": [
   [
    "2026-10-09T06:02:10Z",
    "198.51.100.40"
   ]
  ],
  "unicoop-sul-pix.example": [
   [
    "2026-10-10T06:20:33Z",
    "198.51.100.40"
   ]
  ]
 }
}
''')   # respostas simuladas do caso (plano B)

def crtsh_json(dominio, tentativas=2, espera=5):
    """Consulta https://crt.sh/?q=%.<dominio>&output=json. Devolve lista ou None."""
    q = urllib.parse.quote(f"%.{dominio}")                      # o %25 do original, codificado pelo Python
    url = f"https://crt.sh/?q={q}&output=json"
    for n in range(1, tentativas + 1):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "investigador-cyber-90-dias (estudo)"})
            with urllib.request.urlopen(req, timeout=25) as r:
                corpo = r.read().decode("utf-8", "replace")
            if corpo.lstrip().startswith("["):                  # JSON de verdade começa com [
                return json.loads(corpo)
            print(f"  tentativa {n}: o crt.sh devolveu HTML (sobrecarga?).")
        except Exception as erro:
            print(f"  tentativa {n}: {erro.__class__.__name__}")
        if n < tentativas:
            time.sleep(espera)                                  # espere, não martele
    return None

certs = crtsh_json(DOMINIO_CRT.strip()) if TENTAR_REDE else None
if certs is None:
    certs, origem = RESP["crtsh"], "plano B: crt.sh do caso (fictício, embutido)"
else:
    origem = f"crt.sh pela rede ({DOMINIO_CRT})"
print("Origem:", origem, f"— {len(certs)} certificados\n")

# Equivalente ao jq do original: id, not_before, issuer, common_name, SANs (ordenado e sem repetição)
vistos = set()
for c in sorted(certs, key=lambda c: c["name_value"]):
    sans = ",".join(c["name_value"].split("\n"))
    linha = (c["id"], c["not_before"], c["issuer_name"], c["common_name"], sans[:110] + ("..." if len(sans) > 110 else ""))
    if linha not in vistos:
        vistos.add(linha)
        print("\t".join(map(str, linha)))

### Quem divide certificado com quem?

Um certificado cujo SAN lista vários nomes é **um único pedido** cobrindo todos eles: pivô forte. A não ser que o certificado tenha dezenas de nomes sem relação (estacionamento, hospedagem multi-inquilino).

In [ ]:
LIMITE_SAN = 20   # acima disso, tratamos como certificado multi-inquilino
pares_san, rejeitados_san = [], []
for c in RESP["crtsh"]:
    nomes = sorted({n.removeprefix("www.").removeprefix("painel.").removeprefix("mail.")
                    for n in c["name_value"].split("\n")})
    if len(nomes) > LIMITE_SAN:
        rejeitados_san.append((c["id"], len(nomes)))
        continue
    for i in range(len(nomes)):
        for j in range(i + 1, len(nomes)):
            pares_san.append((nomes[i], nomes[j], c["id"]))

for a, b, cid in pares_san:
    print(f"SAN compartilhado: {a} ↔ {b}  (crt.sh id {cid})")
for cid, n in rejeitados_san:
    print(f"⛔ crt.sh id {cid}: {n} nomes no SAN → certificado multi-inquilino (estacionamento). Rejeitado.")

## 2. Hash de favicon no formato do Shodan, sem `mmh3`

O Shodan calcula `mmh3.hash(base64.encodebytes(icone))`. Abaixo, o MurmurHash3 de 32 bits em Python puro, comentado. Se você tiver `mmh3` instalada (`!pip install mmh3`, opcional), a célula seguinte compara as duas em 300 entradas aleatórias.

In [ ]:
import base64

def murmur3_32(dados: bytes, semente: int = 0) -> int:
    """MurmurHash3 x86 de 32 bits, devolvido como inteiro COM sinal (é o que o Shodan mostra)."""
    c1, c2 = 0xcc9e2d51, 0x1b873593                      # constantes do algoritmo
    h = semente & 0xFFFFFFFF
    def rot(x, r):                                       # rotação à esquerda em 32 bits
        return ((x << r) | (x >> (32 - r))) & 0xFFFFFFFF
    n = len(dados) // 4 * 4                              # parte que cabe em blocos de 4 bytes
    for i in range(0, n, 4):
        k = int.from_bytes(dados[i:i + 4], "little")     # bloco lido em little-endian
        k = rot(k * c1 & 0xFFFFFFFF, 15) * c2 & 0xFFFFFFFF
        h = (rot(h ^ k, 13) * 5 + 0xe6546b64) & 0xFFFFFFFF
    resto = dados[n:]                                    # 0 a 3 bytes que sobraram
    if resto:
        k = int.from_bytes(resto, "little")
        h ^= rot(k * c1 & 0xFFFFFFFF, 15) * c2 & 0xFFFFFFFF
    h ^= len(dados)                                      # mistura final ("fmix")
    h ^= h >> 16
    h = h * 0x85ebca6b & 0xFFFFFFFF
    h ^= h >> 13
    h = h * 0xc2b2ae35 & 0xFFFFFFFF
    h ^= h >> 16
    return h - (1 << 32) if h & 0x80000000 else h       # converte para inteiro com sinal

def favicon_shodan(icone: bytes) -> int:
    # encodebytes quebra linha a cada 76 caracteres, exatamente como o Shodan espera
    return murmur3_32(base64.encodebytes(icone))

# Um "favicon" sintético (bytes inventados), só para ver o cálculo funcionando.
ICONE_SINTETICO = b"\x00\x00\x01\x00" + b"ISCAKIT-FAVICON-SINTETICO-OPERACAO-ISCA|" * 12
print("hash estilo Shodan do ícone sintético:", favicon_shodan(ICONE_SINTETICO))
print("valores de referência do algoritmo: murmur3('hello') =", murmur3_32(b"hello"), "(esperado 613153351)")

In [ ]:
# Opcional: confere contra a biblioteca mmh3, se ela estiver instalada.
!pip -q install mmh3
import os
try:
    import mmh3
    ok = all(mmh3.hash(d) == murmur3_32(d) for d in (os.urandom(n) for n in range(300)))
    print("Confere com mmh3 em 300 entradas aleatórias:", "✅" if ok else "❌")
except ImportError:
    print("mmh3 não instalada: tudo bem, a versão em Python puro já foi conferida pelos autores.")

## 3. Contagem antes de busca (passo 2 do original)

O original manda rodar `shodan count` **antes** de `shodan search`: você vê o tamanho do conjunto antes de gastar créditos e antes de se apegar ao resultado. Abaixo, todas as contagens do caso, passadas pelo teste do fluxograma do Conceito.

In [ ]:
#@title Limites do teste { display-mode: "form" }
LIMITE_REVISAVEL = 25  #@param {type:"integer"}

REGISTRO = []   # o registro de pivôs: cada busca, contagem, decisão e motivo

def avaliar(busca, contagem, forca_base):
    if contagem > 1000:
        decisao, motivo = "rejeitado", "milhares de resultados: valor padrão ou compartilhado"
    elif contagem > LIMITE_REVISAVEL:
        decisao, motivo = "pista", "grande demais para revisar um por um; não conta para convergência"
    else:
        decisao, motivo = "usado", f"{contagem} resultado(s): revisável, força {forca_base}"
    REGISTRO.append({"busca": busca, "contagem": contagem, "decisao": decisao, "motivo": motivo})

FORCA = {"http.favicon.hash:-1029384756": "médio (ícone customizado; o original classifica 'serve favicon' como médio)",
         "ssl.cert.fingerprint": "forte",
         "http.html_hash:1474329905": "médio",
         "ssl.jarm": "fraco", "http.title": "médio", "http.favicon.hash:116323821": "forte só se customizado"}
for busca, n in RESP["shodan_count"].items():
    chave = next((k for k in FORCA if busca.startswith(k)), busca)
    avaliar("Shodan " + busca[:60], n, FORCA.get(chave, "?"))
for busca, r in RESP["urlscan"].items():
    forca_u = ("fraco (IP)" if busca.startswith("page.ip") else
               "forte (recurso do kit)" if busca.startswith("hash:") else "contexto (varredura existente do domínio)")
    avaliar("urlscan " + busca[:60], r["total"], forca_u)
avaliar("crt.sh padrão de nome " + RESP["crtsh_padrao_nome"]["consulta"], RESP["crtsh_padrao_nome"]["resultados"], "fraco")
for cid, n in rejeitados_san:
    REGISTRO.append({"busca": f"crt.sh id {cid} (SAN)", "contagem": n, "decisao": "rejeitado",
                     "motivo": "certificado multi-inquilino de estacionamento"})

# o hash do arquivo vazio é sempre rejeitado, qualquer que seja a contagem
for r in REGISTRO:
    if "e3b0c44298fc1c14" in r["busca"]:
        r["decisao"], r["motivo"] = "rejeitado", "SHA-256 do arquivo VAZIO: aresta sempre falsa"

for r in REGISTRO:
    simbolo = {"usado": "✅", "pista": "🟡", "rejeitado": "⛔"}[r["decisao"]]
    print(f"{simbolo} {r['contagem']:>7}  {r['busca']:<62} {r['motivo']}")

## 4. Nós e arestas (passo 4 do original)

Montamos as arestas a partir dos pivôs **usados** (e das pistas, marcadas como fracas). Certificado, favicon e hash de HTML viram **nós-hub**, como o original recomenda. Depois aplicamos a regra do checkpoint:

> membro = ligado por **dois pivôs independentes**, ou por **um pivô forte com contagem pequena**.

Para saber se dois caminhos são independentes, cada aresta guarda o **tipo de pivô**. Dois caminhos pelo mesmo tipo (ex.: duas resoluções DNS) não contam como dois.

In [ ]:
COLETA = RESP["consultado_em"]
NOS, ARESTAS = {}, []

def neutro(v):
    return "[.]".join(v.rsplit(".", 1))

def no(id_, tipo, rotulo=None):
    NOS.setdefault(id_, {"Id": id_, "Label": rotulo or neutro(id_), "Type": tipo})

def aresta(a, b, pivo, forca, evidencia, contagem):
    ARESTAS.append({"Source": a, "Target": b, "Type": "Undirected", "Label": pivo, "Strength": forca,
                    "Evidence": evidencia, "Count": contagem, "Retrieved": COLETA})

SEMENTES = {"serraazul-pix.example", "203.0.113.77", "lnk.example/sa7"}

# (a) SAN compartilhado (crt.sh): forte, contagem pequena (nº de nomes no certificado)
for a, b, cid in pares_san:
    no(a, "domain"); no(b, "domain")
    aresta(a, b, "SAN de certificado compartilhado", "forte", f"crt.sh id {cid}", 2)

# (b) passive DNS: fraco sozinho (IP compartilhado tem muitos vizinhos)
for dom, resolucoes in RESP["vt_resolutions"].items():
    for quando, ip in resolucoes:
        no(dom, "domain"); no(ip, "ip")
        contagem = RESP["urlscan"].get(f'page.ip:"{ip}"', {}).get("total", 2)
        # IP com 1 ou 2 domínios (os do kit) não é hospedagem compartilhada: sobe para "média"
        forca_dns = "média" if contagem <= 5 else "fraca"
        aresta(dom, ip, "passive DNS", forca_dns, f"VT resolutions {quando[:10]}", contagem)

# (c) Shodan: IP serve favicon / HTML -> hubs
no("fav--1029384756", "favicon", "favicon -1029384756")
no("html-1474329905", "html_hash", "hash HTML 1474329905")
for ip, porta, hosts, titulo in RESP["shodan_search"]["http.favicon.hash:-1029384756"]:
    no(ip, "ip")
    # "serves favicon" é médio no edges.csv do original; ligamos o IP e cada hostname que o Shodan viu nele
    aresta(ip, "fav--1029384756", "serve favicon customizado", "média", f"Shodan favicon search (count=9)", 9)
    for h in hosts:
        no(h, "domain")
        aresta(h, "fav--1029384756", "serve favicon customizado", "média", f"Shodan favicon search, hostname em {ip}", 9)
for ip, porta, hosts, titulo in RESP["shodan_search"]["http.html_hash:1474329905"]:
    for h in hosts:
        no(h, "domain")
        aresta(h, "html-1474329905", "mesmo hash de corpo HTML", "média", "Shodan html_hash (count=3)", 3)

# (d) urlscan: recurso do kit (isca.js) carregado pela página
no("js-c6a500d4", "resource", "isca.js c6a500d4...")
for quando, dom, ip in RESP["urlscan"]["hash:c6a500d420b2dd0197d73e8fa5ebe43f81ebfa3e86e56b91a9e909a4beae08b6"]["amostra"]:
    aresta(dom, "js-c6a500d4", "carrega o mesmo isca.js", "forte", f"urlscan hash: ({quando[:10]})", 2)

# (d2) a string '// pescador' no JS guardado pelo urlscan (artefato de host, Dia 38): média
no("str-pescador", "artefato", "string // pescador no JS")
for dom in ["serraazul-pix.example", "coopvale-pix.example", "credminas-seguranca.example"]:
    aresta(dom, "str-pescador", "mesma string '// pescador' no JS", "média", "cópia do urlscan (não o site ativo)", 3)

# (e) encurtador -> domínio (redirecionamento observado no SMS) e pistas fracas
no("lnk.example/sa7", "url", "hxxps://lnk[.]example/sa7")
aresta("lnk.example/sa7", "serraazul-pix.example", "redirecionamento HTTP observado", "forte", "SMS + captura do Dia 19", 1)
no("coopserra-azul.example", "domain")
aresta("serraazul-pix.example", "coopserra-azul.example", "padrão de nome (sem pivô técnico)", "fraca", "semelhança lexical", 0)
no("cooperouro-regulariza.example", "domain")
aresta("serraazul-pix.example", "cooperouro-regulariza.example", "mesmo título genérico", "fraca", "Shodan http.title (count=214)", 214)

print(f"{len(NOS)} nós, {len(ARESTAS)} arestas.")

In [ ]:
from collections import defaultdict

HUBS = {"fav--1029384756", "html-1474329905", "js-c6a500d4", "str-pescador"}   # propriedades compartilhadas
vizinhos = defaultdict(list)                   # nó -> [(outro nó, aresta)]
for e in ARESTAS:
    vizinhos[e["Source"]].append((e["Target"], e)); vizinhos[e["Target"]].append((e["Source"], e))

# Regra do checkpoint, aplicada até estabilizar (um membro novo pode qualificar outro):
#   membro = 2+ TIPOS de pivô (forte ou média) ligando ao cluster, OU 1 pivô forte com contagem pequena.
cluster, hubs_ok = set(SEMENTES), set()
mudou = True
while mudou:
    mudou = False
    for h in HUBS:                                       # um hub só vale se algo do cluster o serve
        if h not in hubs_ok and any(o in cluster for o, _ in vizinhos[h]):
            hubs_ok.add(h); mudou = True
    for n in NOS:
        if n in cluster or n in HUBS:
            continue
        validas = [(o, e) for o, e in vizinhos[n] if (o in cluster or o in hubs_ok) and e["Strength"] != "fraca"]
        tipos = {e["Label"] for _, e in validas}
        forte_pequena = any(e["Strength"] == "forte" and e["Count"] <= LIMITE_REVISAVEL for _, e in validas)
        if len(tipos) >= 2 or forte_pequena:
            cluster.add(n); mudou = True

def status(n):
    if n in SEMENTES:
        return "semente"
    if n in HUBS:
        return "hub"
    return "membro" if n in cluster else "nao-confirmado"

STATUS = {n: status(n) for n in NOS}
for s in ["semente", "membro", "hub", "nao-confirmado"]:
    print(f"{s:<15}", sorted(neutro(n) if NOS[n]["Type"] in ("domain", "ip") else NOS[n]["Label"]
                              for n, v in STATUS.items() if v == s))

> 💡 Compare com a Prática BR do README e com `caso-continuo/dados/dia-43/cluster-pescador.json` (o mesmo grafo, já pensado para STIX). Os quatro "não confirmados" esperados: `coopserra-azul`, `cooperouro-regulariza`, `192.0.2.150` e `203.0.113.78`. Se o seu resultado divergiu porque você mudou `LIMITE_REVISAVEL`, explique a diferença nas notas: é exatamente a discussão que o checkpoint pede.

In [ ]:
import csv

for n in NOS.values():
    n["Status"] = STATUS[n["Id"]]
with open("nodes.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["Id", "Label", "Type", "Status"]); w.writeheader(); w.writerows(NOS.values())
with open("edges.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["Source", "Target", "Type", "Label", "Strength", "Evidence", "Count", "Retrieved"])
    w.writeheader(); w.writerows(ARESTAS)
with open("registro-de-pivos.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["busca", "contagem", "decisao", "motivo"]); w.writeheader(); w.writerows(REGISTRO)

# Checkpoint do original, por código: nenhuma aresta "relacionado"
assert all(e["Label"] and "relacionad" not in e["Label"].lower() for e in ARESTAS), "Aresta sem pivô específico!"
assert any(r["decisao"] == "rejeitado" for r in REGISTRO), "Falta pelo menos um pivô rejeitado com contagem!"
print("✅ Checkpoints por código ok. 💾 nodes.csv, edges.csv, registro-de-pivos.csv (prontos para o Gephi).")

## 5. O grafo em Mermaid e a camada do ATT&CK (passos 5 e 6)

Antes do Gephi, o desenho. Linha grossa (`===`) = forte, normal (`---`) = média, pontilhada (`-.-`) = fraca; nós não confirmados com borda tracejada, como no original.

In [ ]:
def mid(n):   # Mermaid não aceita pontos e barras em IDs
    return "n_" + re.sub(r"[^A-Za-z0-9]", "_", n)
import re

linhas = ["```mermaid", "graph LR"]
for n, d in NOS.items():
    linhas.append(f'    {mid(n)}["{d["Label"]}<br/>{STATUS[n]}"]')
SETA = {"forte": "===", "média": "---", "fraca": "-.-"}
for e in ARESTAS:
    linhas.append(f'    {mid(e["Source"])} {SETA[e["Strength"]]}|"{e["Label"]} ({e["Count"]})"| {mid(e["Target"])}')
linhas += ["    classDef unconfirmed fill:#ffffff,stroke:#999999,stroke-dasharray:5 5,color:#666666",
           "    classDef hub fill:#e1d5e7,stroke:#9673a6,color:#000000",
           "    classDef seed fill:#dae8fc,stroke:#6c8ebf,color:#000000"]
for s, cls in (("nao-confirmado", "unconfirmed"), ("hub", "hub"), ("semente", "seed")):
    ids = [mid(n) for n, v in STATUS.items() if v == s]
    if ids:
        linhas.append(f"    class {','.join(ids)} {cls}")
linhas.append("```")
MERMAID = "\n".join(linhas)
print(MERMAID)

In [ ]:
#@title Camada de Resource Development { display-mode: "form" }
incluir_T1608_003 = True  #@param {type:"boolean"}
# T1608.003 Install Digital Certificate: o Shodan mostra o certificado SERVIDO na 443 do IP do kit?

tecnicas = [
    {"techniqueID": "T1583.001", "score": 1, "comment": "Domínios imitação registrados dias antes do disparo (RDAP, Dia 39); SAN compartilhado (crt.sh)"},
    {"techniqueID": "T1588.004", "score": 1, "comment": "Certificados de CA pública gratuita obtidos na véspera (crt.sh ids 90000000001, 90000000004)"},
]
if incluir_T1608_003:
    tecnicas.append({"techniqueID": "T1608.003", "score": 1,
                     "comment": "Shodan host 203.0.113.77:443 serve o certificado 9f2c... (instalação observada)"})
camada = {"name": "CA-ISCA-01: Resource Development sustentado pelo grafo",
          "versions": {"layer": "4.5", "attack": "19", "navigator": "5.3.2"},
          "domain": "enterprise-attack",
          "description": "Só técnicas que o grafo do Dia 40 sustenta. ATT&CK v19.2. Dados fictícios (Operação Isca).",
          "techniques": tecnicas,
          "gradient": {"colors": ["#ffffff", "#66b1ff"], "minValue": 0, "maxValue": 1}}
with open("camada-resource-development.json", "w", encoding="utf-8") as f:
    json.dump(camada, f, ensure_ascii=False, indent=1)
with open("grafo-ca-isca-01.md", "w", encoding="utf-8") as f:
    f.write("# Grafo do cluster CA-ISCA-01 (fictício)\n\n" + MERMAID + "\n")
print("💾 camada-resource-development.json e grafo-ca-isca-01.md salvos.")

try:
    from google.colab import files
    for nome in ["nodes.csv", "edges.csv", "registro-de-pivos.csv", "camada-resource-development.json", "grafo-ca-isca-01.md"]:
        files.download(nome)
except ImportError:
    print("(Fora do Colab: os arquivos ficaram na pasta atual.)")

## 6. 🎮 Quiz relâmpago: o pivô vale?

**S** = usa como pivô, **N** = rejeita.

In [ ]:
MODO_JOGO = True
QUIZ = [
    ("Favicon com 9 resultados, ícone customizado do kit", True, "Forte e revisável um por um."),
    ("JARM com 18.544 resultados", False, "Mesma pilha TLS em milhares de servidores."),
    ("hash: e3b0c44298fc...b855 no urlscan", False, "Arquivo vazio: aresta sempre falsa."),
    ("SAN de um certificado com 2 nomes: credminas-seguranca e unicoop-sul-pix", True, "Um pedido, dois nomes: forte."),
    ("SAN de um certificado com 98 nomes de domínios estacionados", False, "Multi-inquilino."),
    ("Título 'Regularização de Chave Pix' com 214 resultados", False, "Título genérico: pista, não pivô."),
    ("Submeter serraazul-pix.example para uma varredura nova no urlscan", False, "Nem é pivô: é tocar o adversário."),
]
acertos = 0
for i, (p, vale, porque) in enumerate(QUIZ, 1):
    print(f"\n[{i}/{len(QUIZ)}] {p}")
    if MODO_JOGO:
        r = ""
        while r not in ("S", "N"):
            r = input("   Usa? (S/N): ").strip().upper()
        ok = (r == "S") == vale
        acertos += ok
        print("   ✅" if ok else "   ❌", end=" ")
    print("USA" if vale else "REJEITA", "—", porque)
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## ✅ Antes de fechar o Dia 40

- [ ] `nodes.csv` e `edges.csv` no Gephi (ForceAtlas 2, cor por `Type`, rótulo das arestas) exportados em PNG/SVG
- [ ] Registro de pivôs com pelo menos um rejeitado e a contagem
- [ ] Membros justificados; não confirmados na cor separada
- [ ] Camada de *Resource Development* só com o que o grafo sustenta
- [ ] Nenhuma consulta sua tocou `serraazul-pix.example` ou os IPs do kit

**Amanhã (Dia 41):** o que é realista monitorar na *dark web*, um Tor seguro, e o plano de coleta que precisa existir **antes** de alguém abrir um fórum criminoso. Rafa já avisou: ninguém vai entrar no grupo onde o IscaKit é vendido.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>